# Notebook 04 — Multi-modal ingest: PDFs and images

**Purpose:** Extend the ingest pipeline from text-only sources (NB 02) to PDFs and images. Demonstrate the §8 file-extension dispatch pattern and the multi-modal flow from §10G.

**Exam relevance:** Models & Capabilities (multi-modal handling) + Architecture Patterns (extension dispatch).
**Design refs:** `docs/marginalia-design.md` §8 (source adapters), §10 Scenario G (local raw folder ingest).
**Depends on:** NB 02 (analyze/synthesize pipeline, retry semantics).

**Fixtures** (committed via git-lfs):
- `data/pdfs/clean.pdf` — 2-page text-rich PDF (matplotlib-generated whitepaper-style).
- `data/pdfs/scanned.pdf` — same content rasterized; no text layer (vision fallback path).
- `data/images/architecture.png` — 4-box system diagram.

Re-generate via `uv run python notebooks/_ops/build_fixtures.py`.


In [ ]:
# Enable autoreload so edits to engine modules are picked up automatically.
%load_ext autoreload
%autoreload 2

import os
import json
import base64
import time
from pathlib import Path
from datetime import date

from dotenv import load_dotenv
from anthropic import Anthropic

from engine.adapters import (
    ExtractedContent,
    extract_image,
    extract_pdf,
    extract_pdf_via_rasterization,
)
from engine.utils.dispatch import extract, UnsupportedExtensionError
from engine.utils.cost_tracker import estimate_cost_usd, record_attempt
from engine.models.pages import SourceKind, PageStatus
from engine.models.wiki_config import MarginaliaConfig
from engine.agents.ingest import analyze_source, synthesize_page


In [ ]:
load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

client = Anthropic()
config = MarginaliaConfig.load(Path("data/poc-wiki"))
print("config loaded:", len(config.purpose_body), "+", len(config.agents_body), "chars")


In [ ]:
# Fixture sanity check — surface a clear error if any are missing.
CLEAN_PDF = Path("data/pdfs/clean.pdf")
SCANNED_PDF = Path("data/pdfs/scanned.pdf")
ARCHITECTURE_PNG = Path("data/images/architecture.png")

for p in (CLEAN_PDF, SCANNED_PDF, ARCHITECTURE_PNG):
    assert p.exists(), (
        f"missing fixture: {p}. Generate with `uv run python notebooks/_ops/build_fixtures.py` "
        "from the repo root."
    )
    print(f"  {p}  ({p.stat().st_size:,} bytes)")

# Render thumbnails for visual reference.
from IPython.display import Image as IPyImage, display
display(IPyImage(filename=str(ARCHITECTURE_PNG), width=480))


## Cell 4 — PDF text path (`pypdf`)

Cheap, deterministic. Every page's text layer is extracted; we report
chars/page so the dispatch heuristic in cell 6 has something to branch on.

In [ ]:
import pypdf
from rich.console import Console
from rich.table import Table

def _per_page_chars(path: Path) -> list[int]:
    reader = pypdf.PdfReader(str(path))
    return [len((page.extract_text() or "")) for page in reader.pages]

text_table = Table(title="pypdf text-layer extraction")
text_table.add_column("fixture")
text_table.add_column("pages", justify="right")
text_table.add_column("total chars", justify="right")
text_table.add_column("chars/page", justify="right")
text_table.add_column("verdict")

for fixture in (CLEAN_PDF, SCANNED_PDF):
    chars = _per_page_chars(fixture)
    total = sum(chars)
    cpp = total / len(chars) if chars else 0
    verdict = "[green]text-dense[/green]" if cpp >= 100 else "[yellow]sparse → vision[/yellow]"
    text_table.add_row(fixture.name, str(len(chars)), str(total), f"{cpp:.0f}", verdict)

Console().print(text_table)

# Show the first page's text from clean.pdf for visual inspection.
clean_first_page = pypdf.PdfReader(str(CLEAN_PDF)).pages[0].extract_text()
print("\n--- clean.pdf page 1 (extracted) ---")
print(clean_first_page[:400])


## Cell 5 — PDF native document content block

Anthropic's native PDF support: send the bytes as a `{"type": "document"}`
block; the model reads scanned PDFs directly without us rasterizing.
This is the path `extract_pdf` falls back to when the text layer is sparse.

In [ ]:
# Run extract_pdf on both fixtures. clean.pdf takes the text path; scanned.pdf falls back.
results_default = {}
for fixture in (CLEAN_PDF, SCANNED_PDF):
    t0 = time.monotonic()
    extracted = extract_pdf(fixture, client=client)
    wall = time.monotonic() - t0
    results_default[fixture.name] = {"extracted": extracted, "wall_s": wall}
    print(f"\n=== {fixture.name} ===")
    print(f"  extraction_method = {extracted.extraction_method}")
    print(f"  chars_per_page    = {extracted.chars_per_page}")
    print(f"  cost_usd          = {extracted.cost_usd}")
    print(f"  wall              = {wall:.2f}s")
    print(f"  text preview      = {extracted.text[:240]!r}")


## Cell 5b — PDF rasterization comparison (manual vision path)

Render each page to PNG via `pypdfium2`, send one image content block
per page in a single Messages call. This is what extension dispatch
*used* to require before native PDF support; documented here for
exam-domain coverage and as the page-level-control alternative.

In [ ]:
# Same PDFs, manual rasterization path. Note the higher token cost.
results_raster = {}
for fixture in (CLEAN_PDF, SCANNED_PDF):
    t0 = time.monotonic()
    extracted = extract_pdf_via_rasterization(fixture, client=client)
    wall = time.monotonic() - t0
    results_raster[fixture.name] = {"extracted": extracted, "wall_s": wall}
    print(f"\n=== {fixture.name} (rasterized) ===")
    print(f"  extraction_method = {extracted.extraction_method}")
    print(f"  cost_usd          = {extracted.cost_usd}")
    print(f"  wall              = {wall:.2f}s")
    print(f"  text preview      = {extracted.text[:240]!r}")


## Cell 6 — Auto-dispatch heuristic

`extract_pdf(path)` is the canonical entry point. It tries `pypdf`;
if the text layer is sparse (<100 chars/page mean, or all whitespace),
it falls back to a native document block. Cell 5 already exercised
this — here we make the branching explicit per fixture.

In [ ]:
dispatch_table = Table(title="extract_pdf — auto-dispatch per fixture")
dispatch_table.add_column("fixture")
dispatch_table.add_column("chars/page", justify="right")
dispatch_table.add_column("threshold", justify="right")
dispatch_table.add_column("branch taken")
dispatch_table.add_column("cost (USD)", justify="right")

for fixture in (CLEAN_PDF, SCANNED_PDF):
    extracted = results_default[fixture.name]["extracted"]
    cpp = extracted.chars_per_page or 0
    branch = extracted.extraction_method
    color = "green" if branch == "text" else "yellow"
    cost_str = "—" if extracted.cost_usd is None else f"${extracted.cost_usd:.6f}"
    dispatch_table.add_row(
        fixture.name,
        f"{cpp:.0f}",
        "100",
        f"[{color}]{branch}[/{color}]",
        cost_str,
    )

Console().print(dispatch_table)


## Cell 7 — Image vision ingest

Single `messages.create` with one image content block. Default model
is Haiku 4.5 — vision-capable and cheap.

In [ ]:
image_extracted = extract_image(ARCHITECTURE_PNG, client=client)
print(f"extraction_method = {image_extracted.extraction_method}")
print(f"cost_usd          = ${image_extracted.cost_usd:.6f}")
print()
print("--- model description ---")
print(image_extracted.text)


## Cell 8 — End-to-end ingest through the existing pipeline

The three extracted texts (clean PDF text, scanned PDF via vision,
architecture image via vision) all flow through the same
`analyze_source` → `synthesize_page` pipeline from NB 02.
This confirms the multi-modal frontend doesn't break the strict-schema
retry contract.

In [ ]:
# Pick the right text source per fixture: clean = pypdf path, scanned = vision_document, image = vision_image.
e2e_inputs = {
    CLEAN_PDF.name: results_default[CLEAN_PDF.name]["extracted"].text,
    SCANNED_PDF.name: results_default[SCANNED_PDF.name]["extracted"].text,
    ARCHITECTURE_PNG.name: image_extracted.text,
}

e2e_results = {}
for name, text in e2e_inputs.items():
    analysis = await analyze_source(text, SourceKind.LOCAL_FILE, config, client=client)
    page, body, log = await synthesize_page(analysis, config, client=client)
    e2e_results[name] = {"analysis": analysis, "page": page, "body": body, "log": log}
    print(f"\n=== {name} ===")
    print(f"  status        = {page.status.value}")
    print(f"  attempts      = {len(log)}")
    print(f"  proposed_type = {analysis.proposed_type.value}")
    print(f"  entities      = {analysis.entities}")


## Cell 9 — Cost comparison across all four extraction paths

The §8 dispatch tradeoff in receipts form: text-only PDF, native
document block, page rasterization, image vision. Costs come from
`engine.utils.cost_tracker.record_attempt` for honesty (no
hand-derived numbers).

In [ ]:
rows = []

# Text-only PDF: no API call.
rows.append({
    "path": "pypdf text",
    "fixture": CLEAN_PDF.name,
    "tokens_in": 0,
    "tokens_out": 0,
    "cost_usd": 0.0,
    "wall_s": None,  # text extraction is microseconds; not worth measuring inline.
})

# Native document block: derived from extract_pdf cost when the vision branch fired.
for fixture in (CLEAN_PDF, SCANNED_PDF):
    extracted = results_default[fixture.name]["extracted"]
    if extracted.extraction_method == "vision_document":
        rows.append({
            "path": "vision_document",
            "fixture": fixture.name,
            "tokens_in": "—",
            "tokens_out": "—",
            "cost_usd": extracted.cost_usd,
            "wall_s": results_default[fixture.name]["wall_s"],
        })

# Rasterization: from cell 5b.
for fixture in (CLEAN_PDF, SCANNED_PDF):
    extracted = results_raster[fixture.name]["extracted"]
    rows.append({
        "path": "vision_rasterized",
        "fixture": fixture.name,
        "tokens_in": "—",
        "tokens_out": "—",
        "cost_usd": extracted.cost_usd,
        "wall_s": results_raster[fixture.name]["wall_s"],
    })

# Image vision: from cell 7.
rows.append({
    "path": "vision_image",
    "fixture": ARCHITECTURE_PNG.name,
    "tokens_in": "—",
    "tokens_out": "—",
    "cost_usd": image_extracted.cost_usd,
    "wall_s": None,
})

cost_table = Table(title="Multi-modal extraction — cost comparison")
cost_table.add_column("path")
cost_table.add_column("fixture")
cost_table.add_column("cost (USD)", justify="right")
cost_table.add_column("wall", justify="right")
for r in rows:
    cost_str = "$0.000000" if r["cost_usd"] == 0.0 else f"${r['cost_usd']:.6f}"
    wall_str = "—" if r["wall_s"] is None else f"{r['wall_s']:.2f}s"
    cost_table.add_row(r["path"], r["fixture"], cost_str, wall_str)

Console().print(cost_table)

# Headline cost ratio: rasterization / document-block on scanned.pdf.
scanned_doc_cost = results_default[SCANNED_PDF.name]["extracted"].cost_usd
scanned_raster_cost = results_raster[SCANNED_PDF.name]["extracted"].cost_usd
if scanned_doc_cost and scanned_raster_cost:
    ratio = scanned_raster_cost / scanned_doc_cost
    print(f"\nrasterization / document_block cost ratio on scanned.pdf: {ratio:.2f}x")


## Cell 10 — Text vs vision: what gets lost or gained

Same PDF (clean.pdf) through two paths: pypdf text and the native
document block (forced via threshold). Diff the resulting analyses
to see what each path emphasizes.

In [ ]:
# Force the vision branch on clean.pdf by lowering the threshold to 0.
# (extract_pdf with text_threshold=0 will *always* take the text branch since
#  even one char clears the bar — so we call the document path directly via
#  a low threshold trick: threshold higher than any plausible cpp.)
clean_via_vision = extract_pdf(CLEAN_PDF, text_threshold=10_000, client=client)
clean_via_text = results_default[CLEAN_PDF.name]["extracted"]
assert clean_via_vision.extraction_method == "vision_document"
assert clean_via_text.extraction_method == "text"

text_analysis = await analyze_source(clean_via_text.text, SourceKind.LOCAL_FILE, config, client=client)
vision_analysis = await analyze_source(clean_via_vision.text, SourceKind.LOCAL_FILE, config, client=client)

text_entities = set(text_analysis.entities)
vision_entities = set(vision_analysis.entities)

console = Console()
diff_table = Table(title="text vs vision_document — same PDF, two paths")
diff_table.add_column("set")
diff_table.add_column("count", justify="right")
diff_table.add_column("members")

diff_table.add_row("both", str(len(text_entities & vision_entities)), ", ".join(sorted(text_entities & vision_entities)) or "—")
diff_table.add_row("[yellow]text only[/yellow]", str(len(text_entities - vision_entities)), ", ".join(sorted(text_entities - vision_entities)) or "—")
diff_table.add_row("[yellow]vision only[/yellow]", str(len(vision_entities - text_entities)), ", ".join(sorted(vision_entities - text_entities)) or "—")

console.print(diff_table)
print("\ntext path summary:   ", text_analysis.summary)
print("\nvision path summary: ", vision_analysis.summary)


## Cell 11 — Receipts: write `engine/decisions/multimodal-dispatch.md`

Parallel to NB 03's `model-selection.md`. Captures the dispatch
decision matrix and cost ratios so future readers can re-evaluate
when prices, models, or PDF characteristics shift.

In [ ]:
DECISIONS_PATH = Path("../engine/decisions/multimodal-dispatch.md")
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)


def _row_for_default(fixture: Path) -> str:
    e = results_default[fixture.name]["extracted"]
    cost = "—" if e.cost_usd is None else f"${e.cost_usd:.6f}"
    cpp = "—" if e.chars_per_page is None else f"{e.chars_per_page:.0f}"
    return f"| {fixture.name} | {e.extraction_method} | {cpp} | {cost} |"


def _row_for_raster(fixture: Path) -> str:
    e = results_raster[fixture.name]["extracted"]
    cost = "—" if e.cost_usd is None else f"${e.cost_usd:.6f}"
    return f"| {fixture.name} | {e.extraction_method} | {cost} |"


scanned_doc = results_default[SCANNED_PDF.name]["extracted"].cost_usd
scanned_raster = results_raster[SCANNED_PDF.name]["extracted"].cost_usd
ratio_line = ""
if scanned_doc and scanned_raster:
    ratio_line = f"Rasterization costs **{scanned_raster / scanned_doc:.1f}×** the native document-block path on `scanned.pdf`."

receipts = f"""# Multi-modal Dispatch Receipts

**Last verified:** {date.today().isoformat()}
**Generated by:** `notebooks/04_multimodal_ingest.ipynb`
**Design ref:** `docs/marginalia-design.md` §8 (extension dispatch), §10G (multi-modal ingest).

## Default path: `extract_pdf`

Tries `pypdf` text extraction first; falls back to a native Anthropic
document content block when `mean(chars/page) < 100` or every page is
whitespace.

| fixture | branch taken | chars/page | cost (USD) |
|---|---|---|---|
{_row_for_default(CLEAN_PDF)}
{_row_for_default(SCANNED_PDF)}

## Comparison path: `extract_pdf_via_rasterization`

Renders each page to PNG via `pypdfium2` and sends one image content
block per page. Documented for transparency and as the page-level
control alternative; not the default.

| fixture | branch taken | cost (USD) |
|---|---|---|
{_row_for_raster(CLEAN_PDF)}
{_row_for_raster(SCANNED_PDF)}

{ratio_line}

## Image path: `extract_image`

| fixture | branch taken | cost (USD) |
|---|---|---|
| {ARCHITECTURE_PNG.name} | vision_image | ${image_extracted.cost_usd:.6f} |

## Dispatch decisions (selected for the engine)

| Extension | Adapter | Default model | Notes |
|---|---|---|---|
| `.md`, `.txt` | `dispatch._extract_text_passthrough` | — | No LLM call. |
| `.pdf` (text-dense) | `extract_pdf` (text branch) | — | `pypdf` only. Free + deterministic. |
| `.pdf` (text-sparse) | `extract_pdf` (vision_document) | claude-sonnet-4-6 | Native document block; one API call. |
| `.pdf` (caller opt-in) | `extract_pdf_via_rasterization` | claude-sonnet-4-6 | Per-page image blocks. Costs more; gives page control. |
| `.png`, `.jpg`, `.jpeg`, `.gif`, `.webp` | `extract_image` | claude-haiku-4-5 | Vision-capable Haiku. |

## Caveats

- Single fixture per path. Widen with real-world PDFs (long, mixed-content) before treating cost ratios as canonical.
- The 100-chars-per-page threshold is unsanity-checked against PDFs with large headers but no body — lower it if you see false-text-positives.
- Native document block has size limits; >32 MB PDFs need rasterization or pre-splitting.
- `temperature` is omitted only for Opus 4.7 (per `engine.utils.api_compat.temperature_kwargs`); all multi-modal calls inherit that compat.

## Re-running

```python
from pathlib import Path
from engine.utils.dispatch import extract

content = extract(Path("notebooks/data/pdfs/clean.pdf"))
print(content.extraction_method, len(content.text), content.cost_usd)
```
"""

DECISIONS_PATH.write_text(receipts, encoding="utf-8")
print(f"wrote {DECISIONS_PATH.resolve()}  ({DECISIONS_PATH.stat().st_size} bytes)")


## What to extract

| Notebook artifact | Extracts to |
|---|---|
| `extract_pdf()` + `extract_pdf_via_rasterization()` | `engine/adapters/local_fs/pdf.py` (extracted) |
| `extract_image()` | `engine/adapters/local_fs/image.py` (extracted) |
| `extract()` extension dispatcher | `engine/utils/dispatch.py` (extracted) |
| `ExtractedContent` adapter contract | `engine/adapters/_template/contract.py` (extracted) |
| Cell 11 receipts | `engine/decisions/multimodal-dispatch.md` (extracted) |

**Notebook-only (intentionally not extracted):**
- The `pypdf` per-page diagnostic table (cell 4).
- The text-vs-vision entity diff (cell 10).
- The cost comparison table (cell 9) — receipts version lives in `multimodal-dispatch.md`.
- The IPython image display calls.

**Fixtures:** committed via git-lfs at `notebooks/data/pdfs/{clean,scanned}.pdf` and `notebooks/data/images/architecture.png`. Regenerate via `uv run python notebooks/_ops/build_fixtures.py`.

**`CACHE_VERSION` discipline:** not triggered by NB 04 (no prompt edits to `ingest_analyze.md` / `ingest_synthesize.md`). The new prompts (`pdf_extract.md`, `image_describe.md`) start at v1; future edits must bump in lockstep with `CACHE_VERSION` once it lands in NB 10.
